In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

SLICE_PATH = "/Users/apple/Desktop/BRAINIAC/slices"
MODEL_PATH = "/Users/apple/Desktop/BRAINIAC/seg_model_2d.h5"

print("Setup loaded")

Setup loaded


In [4]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

TF version: 2.16.2
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import os, glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

In [1]:
SLICE_PATH = "/Users/apple/Desktop/BRAINIAC/slices"
MODEL_PATH = "/Users/apple/Desktop/BRAINIAC/seg_model_2d.h5"

In [6]:
img_files = sorted(glob.glob(f"{SLICE_PATH}/img_*.npy"))
mask_files = sorted(glob.glob(f"{SLICE_PATH}/mask_*.npy"))

print("Total slices:", len(img_files))

Total slices: 20411


In [7]:
def data_generator():
    for img_f, mask_f in zip(img_files, mask_files):
        img = np.load(img_f)
        mask = np.load(mask_f)
        yield img.astype(np.float32), mask[...,None].astype(np.float32)

In [6]:
dataset = tf.data.Dataset.from_generator(
    data_generator,
    output_types=(tf.float32, tf.float32),
    output_shapes=((128,128,3),(128,128,1))
)

dataset = dataset.shuffle(200).batch(8).prefetch(2)

2026-02-21 23:22:49.939753: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-02-21 23:22:49.939933: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-21 23:22:49.939957: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-02-21 23:22:49.940039: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-21 23:22:49.940075: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
def conv_block(x,f):
    x = layers.Conv2D(f,3,padding="same",activation="relu")(x)
    x = layers.Conv2D(f,3,padding="same",activation="relu")(x)
    return x

def build_unet():

    inp = layers.Input((128,128,3))

    c1 = conv_block(inp,16)
    p1 = layers.MaxPool2D()(c1)

    c2 = conv_block(p1,32)
    p2 = layers.MaxPool2D()(c2)

    c3 = conv_block(p2,64)

    u1 = layers.UpSampling2D()(c3)
    u1 = layers.Concatenate()([u1,c2])
    c4 = conv_block(u1,32)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.Concatenate()([u2,c1])
    c5 = conv_block(u2,16)

    out = layers.Conv2D(1,1,activation="sigmoid")(c5)

    return Model(inp,out)

model = build_unet()
model.compile(optimizer="adam",loss="binary_crossentropy")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │      2,320 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │      9,248 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │     36,928 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 64, 64,    │          0 │ conv2d_5[0][0]    │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, 64,    │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 96)               │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 64, 64,    │     27,680 │ concatenate[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 64, 64,    │      9,248 │ conv2d_6[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 128, 128,  │          0 │ conv2d_7[0][0]    │
│ (UpSampling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128, 128,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 48)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 128, 128,  │      6,928 │ concatenate_1[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 128, 128,  │      2,320 │ conv2d_8[0][0]  

 Total params: 118,273 (462.00 KB)

 Trainable params: 118,273 (462.00 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(dataset.take(1000), epochs=3)
model.save(MODEL_PATH)
print("Model saved")

Epoch 1/3


2026-02-21 23:23:30.258126: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 449s 443ms/step - loss: 0.0323
Epoch 2/3


2026-02-21 23:30:58.631377: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-02-21 23:30:58.631410: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 13791267125217218574
2026-02-21 23:30:58.631420: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 17426017024659546482
2026-02-21 23:30:58.631431: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9805788599677290410
2026-02-21 23:30:58.631434: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/adam/Add_20/_62]]
2026-02-21 23:30:58.631446: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 36789142

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 446s 446ms/step - loss: 0.0188
Epoch 3/3


2026-02-21 23:38:25.022232: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-02-21 23:38:25.022254: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10462924472486576294
2026-02-21 23:38:25.022259: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 13791267125217218574
2026-02-21 23:38:25.022272: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 4375385440011269686
2026-02-21 23:38:25.022282: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/adam/Add_12/_94]]
2026-02-21 23:38:25.022304: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 98057885

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 444s 444ms/step - loss: 0.0165


2026-02-21 23:45:49.390916: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-02-21 23:45:49.390934: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10462924472486576294
2026-02-21 23:45:49.390939: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_51]]
2026-02-21 23:45:49.390944: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8283529533449150300
2026-02-21 23:45:49.390948: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14388251947436687389
2026-02-21 23:45:49.390951: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 3347519624933392495
2026-02-

Model saved
